In [8]:
import numpy as np
import tensorflow as tf

In [9]:
npz = np.load('Audiobooks_data_train.npz')
train_inputs = npz['inputs'].astype(np.float32)
train_targets = npz['targets'].astype(np.int32)

npz = np.load('Audiobooks_data_validation.npz')
validation_inputs = npz['inputs'].astype(np.float32)
validation_targets = npz['targets'].astype(np.int32)

npz = np.load('Audiobooks_data_test.npz')
test_inputs = npz['inputs'].astype(np.float32)
test_targets = npz['targets'].astype(np.int32)


In [10]:
mean = np.mean(train_inputs, axis=0)
std = np.std(train_inputs, axis=0)

train_inputs = (train_inputs - mean) / std
validation_inputs = (validation_inputs - mean) / std
test_inputs = (test_inputs - mean) / std


In [11]:
print(np.unique(train_targets, return_counts=True))
print(np.unique(validation_targets, return_counts=True))
print(np.unique(test_targets, return_counts=True))


(array([0, 1], dtype=int32), array([9491, 1776]))
(array([0, 1], dtype=int32), array([1171,  237]))
(array([0, 1], dtype=int32), array([1185,  224]))


In [12]:
input_size = 10
output_size = 2
hidden_layer_size = 200

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu', input_shape=(input_size,),
                          kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                          kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(output_size, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [13]:
batch_size = 50
max_epochs = 100
early_stopping = tf.keras.callbacks.EarlyStopping(
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_inputs, train_targets,
    batch_size=batch_size,
    epochs=max_epochs,
    validation_data=(validation_inputs, validation_targets),
    callbacks=[early_stopping],
    verbose=1
)


Epoch 1/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8785 - loss: 0.5221 - val_accuracy: 0.8942 - val_loss: 0.4318
Epoch 2/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8946 - loss: 0.4096 - val_accuracy: 0.9034 - val_loss: 0.3736
Epoch 3/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8988 - loss: 0.3716 - val_accuracy: 0.8999 - val_loss: 0.3518
Epoch 4/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8987 - loss: 0.3496 - val_accuracy: 0.9034 - val_loss: 0.3297
Epoch 5/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9007 - loss: 0.3315 - val_accuracy: 0.9048 - val_loss: 0.3188
Epoch 6/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9023 - loss: 0.3190 - val_accuracy: 0.9027 - val_loss: 0.3008
Epoch 7/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9035 - loss: 0.3054 - val_accuracy: 0.9027 - val_loss: 0.2989
Epoch 8/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9028 - loss: 0.2967 - val_accu

In [14]:
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print('\nTest loss: {:.2f}. Test accuracy: {:.2f}%'.format(test_loss, test_accuracy * 100))


45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9127 - loss: 0.2471 

Test loss: 0.25. Test accuracy: 91.27%
